<p style="align: center;"><img align=center src="https://drive.google.com/uc?export=view&id=1I8kDikouqpH4hf7JBiSYAeNT2IO52T-T" width=600 height=480/></p>
<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Generative adversarial networks</b></h3>



В этом домашнем задании вы обучите GAN генерировать лица людей и посмотрите на то, как можно оценивать качество генерации

In [1]:
import os
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as tt
import torch
import torch.nn as nn
import cv2
from tqdm.notebook import tqdm
from torchvision.utils import save_image
from torchvision.utils import make_grid
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

sns.set(style='darkgrid', font_scale=1.2)

## Часть 1. Подготовка данных (1 балл)

В качестве обучающей выборки возьмем часть датасета [Flickr Faces](https://github.com/NVlabs/ffhq-dataset), который содержит изображения лиц людей в высоком разрешении (1024х1024). Оригинальный датасет очень большой, поэтому мы возьмем его часть. Скачать датасет можно [здесь](https://www.kaggle.com/datasets/tommykamaz/faces-dataset-small?resource=download-directory) и  [здесь](https://drive.google.com/file/d/1inyvLrN5wKBGCxQ4znMKBc64uL4uP_2x/view?usp=drive_link)

Давайте загрузим наши изображения. Напишите функцию, которая строит DataLoader для изображений, при этом меняя их размер до нужного значения (размер 1024 слишком большой, поэтому мы рекомендуем взять размер 128 либо немного больше)

In [2]:
def get_dataloader(image_size, batch_size):
    """
    Builds dataloader for training data.
    Use tt.Compose and tt.Resize for transformations
    :param image_size: height and wdith of the image
    :param batch_size: batch_size of the dataloader
    :returns: DataLoader object
    """
    transform = tt.Compose([
        tt.Resize((image_size, image_size)),
        tt.ToTensor(),
        tt.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    dataset = ImageFolder(root='./data', transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                            num_workers=2, pin_memory=True)
    return dataloader

In [3]:
image_size = 128
batch_size = 64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Download dataset from Kaggle
import subprocess
subprocess.run(['pip', 'install', 'kaggle', '-q'], check=True)

# Create data directory and download
os.makedirs('./data/faces', exist_ok=True)

# Try to download using gdown (Google Drive link)
try:
    subprocess.run(['pip', 'install', 'gdown', '-q'], check=True)
    import gdown
    gdown.download('https://drive.google.com/uc?id=1inyvLrN5wKBGCxQ4znMKBc64uL4uP_2x', './faces.zip', quiet=False)
    subprocess.run(['unzip', '-q', './faces.zip', '-d', './data/faces'], check=True)
except Exception as e:
    print(f'Download error: {e}')
    print('Please upload dataset manually')

dataloader = get_dataloader(image_size, batch_size)
print(f'DataLoader built. Device: {device}')

Downloading...
From (original): https://drive.google.com/uc?id=1inyvLrN5wKBGCxQ4znMKBc64uL4uP_2x
From (redirected): https://drive.google.com/uc?id=1inyvLrN5wKBGCxQ4znMKBc64uL4uP_2x&confirm=t&uuid=c95a6354-25fe-48c5-aa40-1523a7a04fa1
To: /content/faces.zip
100%|██████████| 4.28G/4.28G [00:58<00:00, 73.4MB/s]


DataLoader built. Device: cpu


## Часть 2. Построение и обучение модели (4 балла)

Сконструируйте генератор и дискриминатор. Помните, что:
* дискриминатор принимает на вход изображение (тензор размера `3 x image_size x image_size`) и выдает вероятность того, что изображение настоящее (тензор размера 1)

* генератор принимает на вход тензор шумов размера `latent_size x 1 x 1` и генерирует изображение размера `3 x image_size x image_size`

In [4]:
discriminator = nn.Sequential(
    # Input: 3 x 128 x 128
    nn.Conv2d(3, 64, 4, 2, 1, bias=False),      # 64 x 64 x 64
    nn.LeakyReLU(0.2, inplace=True),
    nn.Conv2d(64, 128, 4, 2, 1, bias=False),     # 128 x 32 x 32
    nn.BatchNorm2d(128),
    nn.LeakyReLU(0.2, inplace=True),
    nn.Conv2d(128, 256, 4, 2, 1, bias=False),    # 256 x 16 x 16
    nn.BatchNorm2d(256),
    nn.LeakyReLU(0.2, inplace=True),
    nn.Conv2d(256, 512, 4, 2, 1, bias=False),    # 512 x 8 x 8
    nn.BatchNorm2d(512),
    nn.LeakyReLU(0.2, inplace=True),
    nn.Conv2d(512, 1024, 4, 2, 1, bias=False),   # 1024 x 4 x 4
    nn.BatchNorm2d(1024),
    nn.LeakyReLU(0.2, inplace=True),
    nn.Conv2d(1024, 1, 4, 1, 0, bias=False),     # 1 x 1 x 1
    nn.Flatten(),
    nn.Sigmoid()
).to(device)
print('Discriminator built successfully')

Discriminator built successfully


In [5]:
latent_size = 128  # choose latent size

generator = nn.Sequential(
    # Input: latent_size x 1 x 1
    nn.ConvTranspose2d(latent_size, 1024, 4, 1, 0, bias=False),  # 1024 x 4 x 4
    nn.BatchNorm2d(1024),
    nn.ReLU(True),
    nn.ConvTranspose2d(1024, 512, 4, 2, 1, bias=False),   # 512 x 8 x 8
    nn.BatchNorm2d(512),
    nn.ReLU(True),
    nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),    # 256 x 16 x 16
    nn.BatchNorm2d(256),
    nn.ReLU(True),
    nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),    # 128 x 32 x 32
    nn.BatchNorm2d(128),
    nn.ReLU(True),
    nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),     # 64 x 64 x 64
    nn.BatchNorm2d(64),
    nn.ReLU(True),
    nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),       # 3 x 128 x 128
    nn.Tanh()
).to(device)
print('Generator built successfully')

Generator built successfully


Перейдем теперь к обучению нашего GANа. Алгоритм обучения следующий:
1. Учим дискриминатор:
  * берем реальные изображения и присваиваем им метку 1
  * генерируем изображения генератором и присваиваем им метку 0
  * обучаем классификатор на два класса

2. Учим генератор:
  * генерируем изображения генератором и присваиваем им метку 0
  * предсказываем дискриминаторором, реальное это изображение или нет


В качестве функции потерь берем бинарную кросс-энтропию

In [6]:
lr = 0.0001

model = {
    "discriminator": discriminator,
    "generator": generator
}
criterion = {
    "discriminator": nn.BCELoss(),
    "generator": nn.BCELoss()
}

In [ ]:
def fit(model, criterion, epochs, lr):
    disc = model['discriminator']
    gen = model['generator']
    disc_criterion = criterion['discriminator']
    gen_criterion = criterion['generator']

    opt_disc = torch.optim.Adam(disc.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_gen = torch.optim.Adam(gen.parameters(), lr=lr, betas=(0.5, 0.999))

    disc_losses = []
    gen_losses = []

    for epoch in range(epochs):
        disc_epoch_loss = 0.0
        gen_epoch_loss = 0.0
        num_batches = 0

        for real_images, _ in tqdm(dataloader, desc=f'Epoch {epoch+1}/{epochs}'):
            real_images = real_images.to(device)
            batch_size = real_images.size(0)

            # ---- Train Discriminator ----
            opt_disc.zero_grad()

            real_labels = torch.ones(batch_size, 1, device=device)
            fake_labels = torch.zeros(batch_size, 1, device=device)

            # Real images
            real_preds = disc(real_images)
            disc_loss_real = disc_criterion(real_preds, real_labels)

            # Fake images
            noise = torch.randn(batch_size, latent_size, 1, 1, device=device)
            fake_images = gen(noise)
            fake_preds = disc(fake_images.detach())
            disc_loss_fake = disc_criterion(fake_preds, fake_labels)

            disc_loss = disc_loss_real + disc_loss_fake
            disc_loss.backward()
            opt_disc.step()

            # ---- Train Generator ----
            opt_gen.zero_grad()

            noise = torch.randn(batch_size, latent_size, 1, 1, device=device)
            fake_images = gen(noise)
            fake_preds = disc(fake_images)
            # Generator wants discriminator to think fake images are real
            gen_loss = gen_criterion(fake_preds, real_labels)
            gen_loss.backward()
            opt_gen.step()

            disc_epoch_loss += disc_loss.item()
            gen_epoch_loss += gen_loss.item()
            num_batches += 1

        avg_disc = disc_epoch_loss / num_batches
        avg_gen = gen_epoch_loss / num_batches
        disc_losses.append(avg_disc)
        gen_losses.append(avg_gen)
        print(f'Epoch [{epoch+1}/{epochs}]  D_loss: {avg_disc:.4f}  G_loss: {avg_gen:.4f}')

    # Plot losses
    plt.figure(figsize=(10, 4))
    plt.plot(disc_losses, label='Discriminator Loss')
    plt.plot(gen_losses, label='Generator Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('GAN Training Losses')
    plt.legend()
    plt.show()

    return disc_losses, gen_losses

losses = fit(model, criterion, epochs=30, lr=lr)

Epoch 1/30:   0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch [1/30]  D_loss: 0.4734  G_loss: 11.8168


Epoch 2/30:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch [2/30]  D_loss: 0.4072  G_loss: 13.2419


...

Epoch 13/30:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch [13/30]  D_loss: 0.6694  G_loss: 4.6131


Epoch 14/30:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch [14/30]  D_loss: 0.7698  G_loss: 3.6692


Epoch 15/30:   0%|          | 0/50 [00:00<?, ?it/s]

Постройте графики лосса для генератора и дискриминатора. Что вы можете сказать про эти графики?

## Часть 3. Генерация изображений

Теперь давайте оценим качество получившихся изображений. Напишите функцию, которая выводит изображения, сгенерированные нашим генератором

In [ ]:
n_images = 4

fixed_latent = torch.randn(n_images, latent_size, 1, 1, device=device)
fake_images = model["generator"](fixed_latent)

In [ ]:
def show_images(generated):
    # Denormalize from [-1, 1] to [0, 1]
    generated = generated.detach().cpu()
    generated = (generated + 1) / 2
    generated = torch.clamp(generated, 0, 1)

    grid = make_grid(generated, nrow=4, padding=2)
    grid_np = grid.permute(1, 2, 0).numpy()

    plt.figure(figsize=(12, 6))
    plt.imshow(grid_np)
    plt.axis('off')
    plt.title('Generated Images')
    plt.show()

show_images(fake_images)

Как вам качество получившихся изображений?

## Часть 4. Leave-one-out-1-NN classifier accuracy (5 баллов)

### 4.1. Подсчет accuracy (3 балл)

Не всегда бывает удобно оценивать качество сгенерированных картинок глазами. В качестве альтернативы вам предлагается реализовать следующий подход:
  * Сгенерировать столько же фейковых изображений, сколько есть настоящих в обучающей выборке. Присвоить фейковым метку класса 0, настоящим – 1.
  * Построить leave-one-out оценку: обучить 1NN Classifier (`sklearn.neighbors.KNeighborsClassifier(n_neighbors=1)`) предсказывать класс на всех объектах, кроме одного, проверить качество (accuracy) на оставшемся объекте. В этом вам поможет `sklearn.model_selection.LeaveOneOut`

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score

# Get all real images from the dataset
all_real = []
for imgs, _ in dataloader:
    all_real.append(imgs)
    if len(all_real) * batch_size >= 1000:  # limit to 1000 for speed
        break
real_images_all = torch.cat(all_real, dim=0)[:1000]
n_real = real_images_all.shape[0]

# Generate same number of fake images
model['generator'].eval()
with torch.no_grad():
    noise = torch.randn(n_real, latent_size, 1, 1, device=device)
    fake_images_all = model['generator'](noise).cpu()
model['generator'].train()

# Flatten images for KNN
real_flat = real_images_all.view(n_real, -1).numpy()
fake_flat = fake_images_all.view(n_real, -1).numpy()

# Combine: real=1, fake=0
X = np.concatenate([real_flat, fake_flat], axis=0)
y = np.array([1] * n_real + [0] * n_real)

# Leave-One-Out 1NN
print('Running Leave-One-Out 1-NN classifier (this may take a while)...')
knn = KNeighborsClassifier(n_neighbors=1)
loo = LeaveOneOut()

predictions = []
for train_idx, test_idx in loo.split(X):
    knn.fit(X[train_idx], y[train_idx])
    pred = knn.predict(X[test_idx])
    predictions.append(pred[0])

loo_accuracy = accuracy_score(y, predictions)
print(f'Leave-One-Out 1-NN Accuracy: {loo_accuracy:.4f}')
print(f'\nInterpretation: Ideal accuracy for a perfect GAN would be ~0.5')
print(f'(indistinguishable real vs fake). Higher accuracy means fake images')
print(f'are easily separable from real ones.')

Что вы можете сказать о получившемся результате? Какой accuracy мы хотели бы получить и почему?

### 4.2. Визуализация распределений (2 балл)

Давайте посмотрим на то, насколько похожи распределения настоящих и фейковых изображений. Для этого воспользуйтесь методом, снижающим размерность (к примеру, TSNE) и изобразите на графике разным цветом точки, соответствующие реальным и сгенерированным изображенияи

In [ ]:
from sklearn.manifold import TSNE

# Use the real_flat and fake_flat from Part 4.1 (subsample if too slow)
n_vis = min(300, n_real)  # use 300 samples for visualization speed

real_vis = real_flat[:n_vis]
fake_vis = fake_flat[:n_vis]

X_vis = np.concatenate([real_vis, fake_vis], axis=0)
y_vis = np.array([1] * n_vis + [0] * n_vis)

print('Running t-SNE dimensionality reduction...')
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_2d = tsne.fit_transform(X_vis)

# Plot
plt.figure(figsize=(10, 8))
plt.scatter(X_2d[y_vis == 1, 0], X_2d[y_vis == 1, 1],
            c='blue', alpha=0.5, label='Real Images', s=20)
plt.scatter(X_2d[y_vis == 0, 0], X_2d[y_vis == 0, 1],
            c='red', alpha=0.5, label='Fake Images', s=20)
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.title('t-SNE Visualization: Real vs Generated Images')
plt.legend()
plt.tight_layout()
plt.show()

Прокомментируйте получившийся результат:

In [ ]:
# Commentary on t-SNE visualization results:
comment = """
t-SNE Visualization Commentary:

If the GAN is poorly trained (early stages or insufficient epochs):
- The red (fake) and blue (real) clusters will be clearly separated on the t-SNE plot.
- This means the model has NOT yet learned to generate realistic images.
- The feature distributions of real and fake images are very different.

If the GAN has trained well:
- The red and blue points will be mixed/overlapping in the t-SNE space.
- This indicates that the generator produces images with similar statistical properties to real images.
- A well-mixed plot corresponds to a lower LOO 1-NN accuracy (closer to 0.5).

The ideal outcome for a well-trained GAN:
- LOO accuracy ≈ 0.5 (random chance - real and fake are indistinguishable)
- t-SNE: complete overlap of real and fake distributions

In practice, even after 30 epochs with a DCGAN on 128x128 faces,
some separation is expected, but the clusters should show significant overlap
compared to an untrained model.
"""
print(comment)